# Phase 3 -- Production training on Google Colab (T4 GPU)

Local CPU training tops out around the 50K subset. Colab's free T4 makes
the full 590K viable in ~15-30 min for the full 100-epoch budget.

**Workflow:**
1. Upload `data/graphs/hetero.pt` and `data/graphs/features.parquet` from
   the locally-built artefact set to Colab (or rebuild on Colab from a
   smaller raw dump).
2. Mount the repo (clone from GitHub).
3. Install GPU-flavoured torch + torch-geometric + pyg-lib for proper
   `NeighborLoader` performance.
4. Train the GNN, then the ensemble.
5. Download the trained weights to `data/models/{gnn,ensemble}/`.

## 1. Verify GPU + mount Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive  # noqa
drive.mount('/content/drive')

## 2. Clone repo + install

In [ ]:
%cd /content
!rm -rf Meshwatch && git clone https://github.com/ViVas970811/Meshwatch.git
%cd Meshwatch
# CPU torch is already on Colab; install the project + train extras + pyg-lib for GPU NeighborLoader.
!pip install -q -e '.[dev,train]'
!pip install -q pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH_VERSION}+cu121.html

## 3. Copy locally-built Phase 2 artefacts

Path to the artefacts on Drive (adjust to your folder).

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/Meshwatch_artifacts/'
!mkdir -p data/graphs
!cp $DRIVE_DIR/hetero.pt data/graphs/
!cp $DRIVE_DIR/features.parquet data/graphs/

## 4. Stage 1 -- train the GNN on GPU

100 epochs, batch 4096, neighbor sampling [15, 10, 5].

In [ ]:
!python scripts/train.py --epochs 100 --batch-size 4096 --device cuda --num-workers 4

## 5. Stage 2 -- ensemble on GPU + CPU XGBoost

In [ ]:
!python scripts/train_ensemble.py --device cuda

## 6. Save trained artefacts back to Drive

In [ ]:
!cp -r data/models $DRIVE_DIR/

## 7. Sanity-print the Phase-3 acceptance metrics

Plan target: **GNN val AUPRC > 0.65, ensemble val AUPRC > 0.70**.

In [ ]:
import json, pathlib
for p in pathlib.Path('data/models').rglob('val_metrics.json'):
    print(p)
    print(json.dumps(json.loads(p.read_text()), indent=2))
    print('-' * 60)